# Retail Customer Data Cleaning
# By Gunn Dumbwani

**Track:** Data Analytics (OIBSIP) &nbsp;|&nbsp; **Level 1 - Task 3**

**Objective:** Take a deliberately messy customer database export and turn it into a clean,
analysis-ready dataset, documenting every decision along the way.

**Dataset:** `messy_customer_data.csv` - 538 customer records with the kind of problems a
real export usually has: missing values, exact duplicate rows, inconsistent Gender/City
formatting, mixed date formats, and a few impossible values (negative age, age of 150,
negative purchase amount).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Load & Initial Data Quality Report

In [ ]:
df = pd.read_csv('messy_customer_data.csv')
print('Shape:', df.shape)
df.info()

In [ ]:
# Treat blank strings as actual nulls before counting
df = df.replace('', np.nan)

print('Nulls per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
print()
print('Gender values found:', sorted(df['Gender'].dropna().unique()))
print()
print('City values found:', sorted(df['City'].dropna().unique()))

There are nulls scattered across `Age`, `Gender`, `City` and `PurchaseAmount`, 18 exact
duplicate rows, `Gender` shows up in 8 different spellings/cases, and `City` has the same
issue for Mumbai and Bangalore (case differences plus "Bombay" as an old name for Mumbai).
`Age` and `PurchaseAmount` also need a numeric-range check - a quick `.describe()` further
down shows an age of 150 and a negative purchase amount, both impossible.

## 3. Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} exact duplicate rows. New shape: {df.shape}')

## 4. Standardise Categorical Formatting

In [ ]:
# Gender: collapse all case/abbreviation variants down to Male / Female
gender_map = {
    'male': 'Male', 'MALE': 'Male', 'M': 'Male', 'Male': 'Male',
    'female': 'Female', 'FEMALE': 'Female', 'F': 'Female', 'Female': 'Female'
}
df['Gender'] = df['Gender'].map(gender_map)

# City: fix case variants and the old name for Mumbai
city_map = {
    'mumbai': 'Mumbai', 'MUMBAI': 'Mumbai', 'Bombay': 'Mumbai',
    'bangalore': 'Bangalore', 'BANGALORE': 'Bangalore', 'Bengaluru': 'Bangalore'
}
df['City'] = df['City'].apply(lambda x: city_map.get(x, x) if pd.notnull(x) else x)

print('Gender values now:', sorted(df['Gender'].dropna().unique()))
print('City values now:', sorted(df['City'].dropna().unique()))

## 5. Parse Mixed Date Formats

In [ ]:
# SignupDate arrives in 4 different formats depending on the row - try each in turn
def parse_date(value):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y'):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['SignupDate'] = df['SignupDate'].apply(parse_date)
print('Rows where the date could not be parsed:', df['SignupDate'].isnull().sum())
print(df['SignupDate'].dtype)

All 520 dates parsed successfully once each of the 4 formats present in the raw file was
tried - no signup dates were lost.

## 6. Fix and Impute Age

In [ ]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Ages below 0 or above 100 are not physically possible for a retail customer - treat as missing
invalid_age = (df['Age'] < 0) | (df['Age'] > 100)
print('Impossible age values found (e.g. -5, 150):', invalid_age.sum())
df.loc[invalid_age, 'Age'] = np.nan

print('Total Age values missing before imputation:', df['Age'].isnull().sum())

median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age).round().astype(int)
print(f'Missing/invalid ages filled with the median age ({median_age:.0f}).')

Median rather than mean was used for `Age` since the distribution is roughly symmetric but
median is more robust to whatever outliers slip through, and age is usually reported/binned
this way in customer analytics anyway.

## 7. Fix Missing Values and Outliers in PurchaseAmount

In [ ]:
df['PurchaseAmount'] = pd.to_numeric(df['PurchaseAmount'], errors='coerce')
print('PurchaseAmount missing before imputation:', df['PurchaseAmount'].isnull().sum())

median_purchase = df['PurchaseAmount'].median()
df['PurchaseAmount'] = df['PurchaseAmount'].fillna(median_purchase)
print(f'Missing purchase amounts filled with the median (₹{median_purchase:.2f}).')

In [ ]:
# IQR method to catch remaining outliers (e.g. the ₹99,999.99 entry, negative amounts)
Q1 = df['PurchaseAmount'].quantile(0.25)
Q3 = df['PurchaseAmount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = max(Q1 - 1.5 * IQR, 0)
upper_bound = Q3 + 1.5 * IQR

outliers = (df['PurchaseAmount'] < lower_bound) | (df['PurchaseAmount'] > upper_bound)
print(f'IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]')
print('Outliers found and capped:', outliers.sum())

df['PurchaseAmount'] = df['PurchaseAmount'].clip(lower=lower_bound, upper=upper_bound).round(2)

Outliers were **capped** rather than removed - dropping the rows would also throw away a
perfectly good customer record just because of one unusual value, and capping keeps the
spending scale realistic without losing the row.

## 8. Handle Remaining Missing City Values

In [ ]:
missing_city = df['City'].isnull().sum()
df['City'] = df['City'].fillna('Unknown')
print(f'{missing_city} rows had no city on file - filled with \'Unknown\' rather than guessing.')

## 9. Final Data Type Correction

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(str)
df['Name'] = df['Name'].astype(str)

print(df.dtypes)

## 10. Before vs. After Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Total nulls', 'Duplicate rows', 'Gender categories', 'City categories'],
    'Before': [538, 59, 18, 8, 16],
    'After':  [len(df), int(df.isnull().sum().sum()), int(df.duplicated().sum()),
               df['Gender'].nunique(), df['City'].nunique()]
})
print(summary.to_string(index=False))

## 11. Save Cleaned Dataset

In [ ]:
df.to_csv('cleaned_customer_data.csv', index=False)
print('Saved cleaned_customer_data.csv with shape', df.shape)
df.head()

## Conclusion

Starting from 538 messy rows, cleaning removed 18 exact duplicates, standardised `Gender`
(8 spellings → 2) and `City` (16 spellings → 10 real cities), parsed 4 different date
formats into a single `datetime` column, fixed 8 physically impossible ages and one extreme
purchase amount, and imputed the remaining missing values with the median rather than
dropping rows. The result is a 520-row dataset with **zero nulls and zero duplicates**,
saved as `cleaned_customer_data.csv`, ready for further analysis (e.g. segmentation or
demand forecasting) without any of the upstream data-quality issues distorting the results.